In [2]:
# Setup cell — run this first. Do not modify.

# Install the Snowflake connector if the environment does not have it yet
%pip install -q snowflake-connector-python

# Import the libraries used in this task
import pandas as pd
from decimal import Decimal
from getpass import getpass
import snowflake.connector


# Database made in the Lab 9 Task
DB = "SA_AIR_TASK"

# Ask for your Snowflake details (nothing is stored in the notebook)
sf_account = input("Snowflake account identifier: ")
sf_user = input("Snowflake user name: ")
sf_password = getpass("Snowflake password: ")
sf_warehouse = input("Snowflake warehouse [SNOWFLAKE_LEARNING_WH]: ") or "SNOWFLAKE_LEARNING_WH"

# Open the connection
conn = snowflake.connector.connect(
    account=sf_account,
    user=sf_user,
    password=sf_password,
    warehouse=sf_warehouse,
)


def run_sql(sql):
    """Run ONE SQL statement in Snowflake and return the result as a DataFrame."""
    cur = conn.cursor()
    try:
        # Execute the statement
        cur.execute(sql)
        # Statements without a result set (for example CREATE) return an empty DataFrame
        if cur.description is None:
            return pd.DataFrame()
        # Fetch the rows and use the column names that Snowflake returns
        df = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
    finally:
        cur.close()
    # Snowflake returns decimal numbers as Decimal objects, so convert them to float
    for col in df.columns:
        if df[col].map(lambda v: isinstance(v, Decimal)).any():
            df[col] = df[col].astype(float)
    return df


# Check that the tables of the Lab 9 Task exist and are not empty
try:
    lab9_tables = run_sql(f"""
    SELECT 'MART.DIM_COUNTRY' AS TABLE_NAME, COUNT(*) AS TOTAL_ROWS FROM {DB}.MART.DIM_COUNTRY
    UNION ALL SELECT 'MART.DIM_LOCATION', COUNT(*) FROM {DB}.MART.DIM_LOCATION
    UNION ALL SELECT 'MART.DIM_DATETIME', COUNT(*) FROM {DB}.MART.DIM_DATETIME
    UNION ALL SELECT 'MART.FACT_AIR_HOURLY', COUNT(*) FROM {DB}.MART.FACT_AIR_HOURLY
    """)
    assert (lab9_tables["TOTAL_ROWS"] > 0).all(), "A Lab 9 Task table is empty"
    print(lab9_tables.to_string(index=False))
    print("Setup finished. The Lab 9 Task tables are ready.")
except Exception as error:
    raise RuntimeError("The Lab 9 Task tables are missing or empty. Run your Lab 9 Task notebook first.") from error



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Snowflake account identifier: YSYOVZQ-QQ10633
Snowflake user name: 233670
Snowflake password: ******
Snowflake warehouse [SNOWFLAKE_LEARNING_WH]: SNOWFLAKE_LEARNING_WH
          TABLE_NAME  TOTAL_ROWS
    MART.DIM_COUNTRY           3
   MART.DIM_LOCATION           4
   MART.DIM_DATETIME          72
MART.FACT_AIR_HOURLY         288
Setup finished. The Lab 9 Task tables are ready.


In [3]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.3 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Create and check the flat view

# 1. Join the fact table to all three dimensions.
#    CASE creates the new time-hierarchy level HOUR_BLOCK from HOUR_NUMBER.
run_sql(f"""
CREATE OR REPLACE VIEW {DB}.MART.V_AIR_HOURLY AS
SELECT
    c.COUNTRY AS COUNTRY,
    l.CITY AS CITY,
    d.DATE_VALUE AS DATE_VALUE,
    d.DAY_NAME AS DAY_NAME,
    d.HOUR_NUMBER AS HOUR_NUMBER,
    CASE
        WHEN d.HOUR_NUMBER BETWEEN 0 AND 5 THEN '00-05'
        WHEN d.HOUR_NUMBER BETWEEN 6 AND 11 THEN '06-11'
        WHEN d.HOUR_NUMBER BETWEEN 12 AND 17 THEN '12-17'
        WHEN d.HOUR_NUMBER BETWEEN 18 AND 23 THEN '18-23'
    END AS HOUR_BLOCK,
    f.PM2_5 AS PM2_5,
    f.PM10 AS PM10,
    f.NITROGEN_DIOXIDE AS NITROGEN_DIOXIDE,
    f.OZONE AS OZONE,
    f.US_AQI AS US_AQI
FROM {DB}.MART.FACT_AIR_HOURLY AS f
JOIN {DB}.MART.DIM_LOCATION AS l
    ON f.LOCATION_KEY = l.LOCATION_KEY
JOIN {DB}.MART.DIM_COUNTRY AS c
    ON l.COUNTRY_KEY = c.COUNTRY_KEY
JOIN {DB}.MART.DIM_DATETIME AS d
    ON f.DATETIME_KEY = d.DATETIME_KEY
""")

print("View created:", f"{DB}.MART.V_AIR_HOURLY")

# 2. The view must contain exactly as many rows as the fact table.
view_count = int(run_sql(
    f"SELECT COUNT(*) AS N FROM {DB}.MART.V_AIR_HOURLY"
).iloc[0]["N"])

fact_count = int(run_sql(
    f"SELECT COUNT(*) AS N FROM {DB}.MART.FACT_AIR_HOURLY"
).iloc[0]["N"])

print("View rows:", view_count)
print("Fact table rows:", fact_count)
assert view_count == fact_count, "The joins have lost or duplicated fact rows."

# 3. Check required dimension values and the four hour blocks in one query.
view_check = run_sql(f"""
SELECT
    COUNT_IF(COUNTRY IS NULL) AS NULL_COUNTRY,
    COUNT_IF(CITY IS NULL) AS NULL_CITY,
    COUNT_IF(DATE_VALUE IS NULL) AS NULL_DATE,
    COUNT_IF(HOUR_BLOCK IS NULL) AS NULL_HOUR_BLOCK,
    COUNT(DISTINCT HOUR_BLOCK) AS DISTINCT_HOUR_BLOCKS
FROM {DB}.MART.V_AIR_HOURLY
""")
display(view_check)

assert int(view_check.iloc[0]["NULL_COUNTRY"]) == 0
assert int(view_check.iloc[0]["NULL_CITY"]) == 0
assert int(view_check.iloc[0]["NULL_DATE"]) == 0
assert int(view_check.iloc[0]["NULL_HOUR_BLOCK"]) == 0
assert int(view_check.iloc[0]["DISTINCT_HOUR_BLOCKS"]) == 4

# 4. Read the complete view into air_df for Tasks 10.2–10.5.
air_df = run_sql(f"SELECT * FROM {DB}.MART.V_AIR_HOURLY")

print("air_df shape (rows, columns):", air_df.shape)
print("Distinct countries:", air_df["COUNTRY"].nunique())
print("Distinct cities:", air_df["CITY"].nunique())
print("Distinct dates:", air_df["DATE_VALUE"].nunique())
print("Distinct hour blocks:", air_df["HOUR_BLOCK"].nunique())

display(air_df.head())

View created: SA_AIR_TASK.MART.V_AIR_HOURLY
View rows: 288
Fact table rows: 288


,NULL_COUNTRY,NULL_CITY,NULL_DATE,NULL_HOUR_BLOCK,DISTINCT_HOUR_BLOCKS
0,0,0,0,0,4


air_df shape (rows, columns): (288, 11)
Distinct countries: 3
Distinct cities: 4
Distinct dates: 3
Distinct hour blocks: 4


,COUNTRY,CITY,DATE_VALUE,DAY_NAME,HOUR_NUMBER,HOUR_BLOCK,PM2_5,PM10,NITROGEN_DIOXIDE,OZONE,US_AQI
0,Pakistan,Peshawar,2026-11-11,Wed,0,00-05,46.4,51.7,36.7,8.0,107.0
1,Pakistan,Peshawar,2026-11-11,Wed,1,00-05,46.1,51.2,36.6,4.0,107.0
2,Pakistan,Peshawar,2026-11-11,Wed,2,00-05,54.1,59.4,36.1,0.0,108.0
3,Pakistan,Peshawar,2026-11-11,Wed,3,00-05,64.0,69.2,33.4,6.0,108.0
4,Pakistan,Peshawar,2026-11-11,Wed,4,00-05,43.9,51.6,26.5,35.0,110.0


In [11]:
# Roll-up along the hierarchy

# Helper: create one aggregation level in Snowflake.
# An empty group_cols list means one grand-total row.
ALLOWED_GROUPS = {"COUNTRY", "CITY", "DATE_VALUE", "HOUR_BLOCK"}

def level(group_cols):
    for col in group_cols:
        if col not in ALLOWED_GROUPS:
            raise ValueError(f"Unsupported grouping column: {col}")

    select_groups = ", ".join(group_cols)
    group_clause = f"GROUP BY {select_groups}" if group_cols else ""

    if group_cols:
        sql = f"""
        SELECT
            {select_groups},
            SUM(PM2_5) AS SUM_PM2_5,
            COUNT(*) AS HOURLY_ROWS
        FROM {DB}.MART.V_AIR_HOURLY
        {group_clause}
        ORDER BY {select_groups}
        """
    else:
        sql = f"""
        SELECT
            SUM(PM2_5) AS SUM_PM2_5,
            COUNT(*) AS HOURLY_ROWS
        FROM {DB}.MART.V_AIR_HOURLY
        """

    return run_sql(sql)

# Level 0 is the most detailed level. Each row is one city/date/hour block.
base_df = level(["COUNTRY", "CITY", "DATE_VALUE", "HOUR_BLOCK"])

# Each next level removes one grouping column.
city_date_df = level(["COUNTRY", "CITY", "DATE_VALUE"])
city_df = level(["COUNTRY", "CITY"])
country_df = level(["COUNTRY"])
total_df = level([])

print("Level 0 — Country, City, Date, Hour block")
display(base_df)
print("Level 1 — Country, City, Date")
display(city_date_df)
print("Level 2 — Country, City")
display(city_df)
print("Level 3 — Country")
display(country_df)
print("Level 4 — Grand total")
display(total_df)

# Helper: compare a SQL roll-up level with an aggregation of base_df.
# The base_df is the only source used for these pandas roll-up checks.
def check_rollup_against_base(sql_level_df, group_cols):
    if group_cols:
        pandas_level = (
            base_df.groupby(group_cols, as_index=False, dropna=False)
            .agg(
                SUM_PM2_5=("SUM_PM2_5", "sum"),
                HOURLY_ROWS=("HOURLY_ROWS", "sum")
            )
        )
        merged = sql_level_df.merge(
            pandas_level,
            on=group_cols,
            how="outer",
            suffixes=("_SQL", "_PANDAS"),
            indicator=True
        )
        assert (merged["_merge"] == "both").all(), "The groups do not match."
    else:
        pandas_level = pd.DataFrame([{
            "SUM_PM2_5": base_df["SUM_PM2_5"].sum(),
            "HOURLY_ROWS": base_df["HOURLY_ROWS"].sum()
        }])
        merged = sql_level_df.merge(
            pandas_level,
            how="cross",
            suffixes=("_SQL", "_PANDAS")
        )

    assert (merged["HOURLY_ROWS_SQL"] == merged["HOURLY_ROWS_PANDAS"]).all()
    assert (
        (merged["SUM_PM2_5_SQL"] - merged["SUM_PM2_5_PANDAS"]).abs() < 0.01
    ).all()
    print(f"Pandas roll-up check passed for {group_cols or 'grand total'}.")

# Compare levels 1 to 4 with pandas calculations from base_df.
check_rollup_against_base(city_date_df, ["COUNTRY", "CITY", "DATE_VALUE"])
check_rollup_against_base(city_df, ["COUNTRY", "CITY"])
check_rollup_against_base(country_df, ["COUNTRY"])
check_rollup_against_base(total_df, [])

# Check that each SQL level has the expected number of distinct groups in air_df.
level_checks = [
    (base_df, ["COUNTRY", "CITY", "DATE_VALUE", "HOUR_BLOCK"]),
    (city_date_df, ["COUNTRY", "CITY", "DATE_VALUE"]),
    (city_df, ["COUNTRY", "CITY"]),
    (country_df, ["COUNTRY"])
]

for result_df, group_cols in level_checks:
    expected_groups = len(air_df[group_cols].drop_duplicates())
    assert len(result_df) == expected_groups, (
        f"Wrong number of groups for {group_cols}: "
        f"{len(result_df)} != {expected_groups}"
    )

assert int(total_df.iloc[0]["HOURLY_ROWS"]) == len(air_df)
print("All level row-count checks passed.")

# Average is not additive. Calculate it from SUM / COUNT, not by summing averages.
country_df["AVG_PM2_5"] = (
    country_df["SUM_PM2_5"] / country_df["HOURLY_ROWS"]
).round(2)

country_average_pandas = (
    air_df.groupby("COUNTRY")["PM2_5"].mean().round(2)
)
for _, row in country_df.iterrows():
    expected_average = country_average_pandas.loc[row["COUNTRY"]]
    assert abs(row["AVG_PM2_5"] - expected_average) < 0.011

print("Country averages calculated from SUM_PM2_5 / HOURLY_ROWS:")
display(country_df)

# One SQL ROLLUP query. GROUPING() is 0 for a normal value and 1 for a subtotal.
rollup_df = run_sql(f"""
SELECT
    COUNTRY,
    CITY,
    DATE_VALUE,
    HOUR_BLOCK,
    SUM(PM2_5) AS SUM_PM2_5,
    COUNT(*) AS HOURLY_ROWS,
    GROUPING(COUNTRY) AS G_COUNTRY,
    GROUPING(CITY) AS G_CITY,
    GROUPING(DATE_VALUE) AS G_DATE,
    GROUPING(HOUR_BLOCK) AS G_BLOCK
FROM {DB}.MART.V_AIR_HOURLY
GROUP BY ROLLUP (COUNTRY, CITY, DATE_VALUE, HOUR_BLOCK)
""")

expected_rollup_rows = (
    1 + len(country_df) + len(city_df) +
    len(city_date_df) + len(base_df)
)
assert len(rollup_df) == expected_rollup_rows

print("Number of ROLLUP rows:", len(rollup_df))
print("Rows at each ROLLUP level (identified by GROUPING flags):")
rollup_level_counts = (
    rollup_df.groupby(
        ["G_COUNTRY", "G_CITY", "G_DATE", "G_BLOCK"]
    ).size().reset_index(name="ROW_COUNT")
)
display(rollup_level_counts)

# Country subtotal rows have a real country and NULLs for lower levels.
rollup_country_df = rollup_df[
    (rollup_df["G_COUNTRY"] == 0) &
    (rollup_df["G_CITY"] == 1) &
    (rollup_df["G_DATE"] == 1) &
    (rollup_df["G_BLOCK"] == 1)
][["COUNTRY", "SUM_PM2_5", "HOURLY_ROWS"]]

country_compare = country_df[
    ["COUNTRY", "SUM_PM2_5", "HOURLY_ROWS"]
].merge(
    rollup_country_df,
    on="COUNTRY",
    how="outer",
    suffixes=("_LEVEL", "_ROLLUP"),
    indicator=True
)

assert (country_compare["_merge"] == "both").all()
assert (
    country_compare["HOURLY_ROWS_LEVEL"] ==
    country_compare["HOURLY_ROWS_ROLLUP"]
).all()
assert (
    (country_compare["SUM_PM2_5_LEVEL"] -
     country_compare["SUM_PM2_5_ROLLUP"]).abs() < 0.01
).all()

Level 0 — Country, City, Date, Hour block


,COUNTRY,CITY,DATE_VALUE,HOUR_BLOCK,SUM_PM2_5,HOURLY_ROWS
0,Bangladesh,Dhaka,2026-11-11,00-05,173.1,6
1,Bangladesh,Dhaka,2026-11-11,06-11,139.7,6
2,Bangladesh,Dhaka,2026-11-11,12-17,349.5,6
3,Bangladesh,Dhaka,2026-11-11,18-23,454.0,6
4,Bangladesh,Dhaka,2026-11-12,00-05,308.2,6
5,Bangladesh,Dhaka,2026-11-12,06-11,179.3,6
6,Bangladesh,Dhaka,2026-11-12,12-17,455.1,6
7,Bangladesh,Dhaka,2026-11-12,18-23,576.0,6
8,Bangladesh,Dhaka,2026-11-13,00-05,434.4,6
9,Bangladesh,Dhaka,2026-11-13,06-11,208.8,6


Level 1 — Country, City, Date


,COUNTRY,CITY,DATE_VALUE,SUM_PM2_5,HOURLY_ROWS
0,Bangladesh,Dhaka,2026-11-11,1116.3,24
1,Bangladesh,Dhaka,2026-11-12,1518.6,24
2,Bangladesh,Dhaka,2026-11-13,1555.7,24
3,Pakistan,Peshawar,2026-11-11,1244.1,24
4,Pakistan,Peshawar,2026-11-12,1313.3,24
5,Pakistan,Peshawar,2026-11-13,1823.2,24
6,Pakistan,Quetta,2026-11-11,646.6,24
7,Pakistan,Quetta,2026-11-12,649.0,24
8,Pakistan,Quetta,2026-11-13,308.4,24
9,Sri Lanka,Colombo,2026-11-11,271.0,24


Level 2 — Country, City


,COUNTRY,CITY,SUM_PM2_5,HOURLY_ROWS
0,Bangladesh,Dhaka,4190.6,72
1,Pakistan,Peshawar,4380.6,72
2,Pakistan,Quetta,1604.0,72
3,Sri Lanka,Colombo,1122.5,72


... (4 outputs left out)

Pandas roll-up check passed for ['COUNTRY', 'CITY', 'DATE_VALUE'].
Pandas roll-up check passed for ['COUNTRY', 'CITY'].
Pandas roll-up check passed for ['COUNTRY'].
Pandas roll-up check passed for grand total.
All level row-count checks passed.
Country averages calculated from SUM_PM2_5 / HOURLY_ROWS:


,COUNTRY,SUM_PM2_5,HOURLY_ROWS,AVG_PM2_5
0,Bangladesh,4190.6,72,58.20
1,Pakistan,5984.6,144,41.56
2,Sri Lanka,1122.5,72,15.59


Number of ROLLUP rows: 68
Rows at each ROLLUP level (identified by GROUPING flags):


,G_COUNTRY,G_CITY,G_DATE,G_BLOCK,ROW_COUNT
0,0,0,0,0,48
1,0,0,0,1,12
2,0,0,1,1,4
3,0,1,1,1,3
4,1,1,1,1,1


In [12]:
# Drill down from country to the peak hour

# Only these columns are allowed as drill-down levels.
DRILL_COLUMNS = {"COUNTRY", "CITY", "DATE_VALUE", "HOUR_BLOCK", "HOUR_NUMBER"}

def sql_text(value):
    """Safely quote a text value for a SQL string literal."""
    return "'" + str(value).replace("'", "''") + "'"

def date_text(value):
    """Convert a Snowflake/Pandas date value to YYYY-MM-DD text."""
    return pd.to_datetime(value).strftime("%Y-%m-%d")

def drill(group_col, filters):
    if group_col not in DRILL_COLUMNS:
        raise ValueError(f"Unsupported drill column: {group_col}")

    where_clause = ""
    if filters:
        where_clause = "WHERE " + " AND ".join(filters)

    return run_sql(f"""
    SELECT
        {group_col},
        ROUND(AVG(US_AQI), 2) AS AVG_US_AQI,
        COUNT(*) AS HOURLY_ROWS
    FROM {DB}.MART.V_AIR_HOURLY
    {where_clause}
    GROUP BY {group_col}
    ORDER BY AVG(US_AQI) DESC
    """)

# Step 1: find the country with the highest average US AQI.
dd_country_df = drill("COUNTRY", [])
top_country = dd_country_df.iloc[0]["COUNTRY"]
display(dd_country_df)

# Step 2: use the country selected at Step 1; do not type a country manually.
dd_city_df = drill(
    "CITY",
    [f"COUNTRY = {sql_text(top_country)}"]
)
top_city = dd_city_df.iloc[0]["CITY"]
display(dd_city_df)

# Step 3: use the city selected at Step 2.
dd_date_df = drill(
    "DATE_VALUE",
    [f"CITY = {sql_text(top_city)}"]
)
top_date = date_text(dd_date_df.iloc[0]["DATE_VALUE"])
display(dd_date_df)

# Step 4: use both the selected city and selected date.
dd_block_df = drill(
    "HOUR_BLOCK",
    [
        f"CITY = {sql_text(top_city)}",
        f"DATE_VALUE = TO_DATE({sql_text(top_date)})"
    ]
)
top_block = dd_block_df.iloc[0]["HOUR_BLOCK"]
display(dd_block_df)

# Step 5: use the same city/date and the hour block selected at Step 4.
dd_hour_df = drill(
    "HOUR_NUMBER",
    [
        f"CITY = {sql_text(top_city)}",
        f"DATE_VALUE = TO_DATE({sql_text(top_date)})",
        f"HOUR_BLOCK = {sql_text(top_block)}"
    ]
)
peak_hour = int(dd_hour_df.iloc[0]["HOUR_NUMBER"])
peak_hour_aqi = float(dd_hour_df.iloc[0]["AVG_US_AQI"])
display(dd_hour_df)

# Pandas filters for the number-of-records checks at each step.
date_series = pd.to_datetime(air_df["DATE_VALUE"]).dt.strftime("%Y-%m-%d")
selected_step_rows = [
    air_df,
    air_df[air_df["COUNTRY"] == top_country],
    air_df[air_df["CITY"] == top_city],
    air_df[
        (air_df["CITY"] == top_city) &
        (date_series == top_date)
    ],
    air_df[
        (air_df["CITY"] == top_city) &
        (date_series == top_date) &
        (air_df["HOUR_BLOCK"] == top_block)
    ]
]
drill_results = [
    dd_country_df, dd_city_df, dd_date_df, dd_block_df, dd_hour_df
]

for step_number, (result_df, selected_rows) in enumerate(
    zip(drill_results, selected_step_rows), start=1
):
    assert int(result_df["HOURLY_ROWS"].sum()) == len(selected_rows), (
        f"Drill step {step_number}: row counts do not add up."
    )

# The weighted average of each lower level must equal the selected value
# at the previous level. Weighting is necessary because groups may have
# different numbers of hourly records.
def weighted_average(result_df):
    return (
        (result_df["AVG_US_AQI"] * result_df["HOURLY_ROWS"]).sum()
        / result_df["HOURLY_ROWS"].sum()
    )

assert abs(weighted_average(dd_city_df) -
           float(dd_country_df.iloc[0]["AVG_US_AQI"])) < 0.011
assert abs(weighted_average(dd_date_df) -
           float(dd_city_df.iloc[0]["AVG_US_AQI"])) < 0.011
assert abs(weighted_average(dd_block_df) -
           float(dd_date_df.iloc[0]["AVG_US_AQI"])) < 0.011
assert abs(weighted_average(dd_hour_df) -
           float(dd_block_df.iloc[0]["AVG_US_AQI"])) < 0.011

drill_path = (
    f"Country: {top_country} > City: {top_city} > "
    f"Date: {top_date} > Hour block: {top_block} > "
    f"Hour: {peak_hour}:00 UTC (US AQI: {peak_hour_aqi})"
)
print(drill_path)

drill_note = (
    f"The drill-down identifies the highest-average US AQI path from "
    f"{top_country} to {top_city}, then to {top_date}, {top_block}, "
    f"and hour {peak_hour}:00 UTC. It shows where and when the forecast "
    f"US AQI is highest within this small dataset. These values are model "
    f"forecasts for a short period, not year-round measurements from "
    f"ground-based air-quality sensors."
)
print(drill_note)

,COUNTRY,AVG_US_AQI,HOURLY_ROWS
0,Bangladesh,143.71,72
1,Pakistan,111.29,144
2,Sri Lanka,63.65,72


,CITY,AVG_US_AQI,HOURLY_ROWS
0,Dhaka,143.71,72


,DATE_VALUE,AVG_US_AQI,HOURLY_ROWS
0,2026-11-13,159.71,24
1,2026-11-12,146.00,24
2,2026-11-11,125.42,24


,HOUR_BLOCK,AVG_US_AQI,HOURLY_ROWS
0,12-17,161.17,6
1,06-11,159.83,6
2,18-23,159.83,6
3,00-05,158.00,6


,HOUR_NUMBER,AVG_US_AQI,HOURLY_ROWS
0,12,164.0,1
1,15,161.0,1
2,16,161.0,1
3,17,161.0,1
4,13,160.0,1
5,14,160.0,1


Country: Bangladesh > City: Dhaka > Date: 2026-11-13 > Hour block: 12-17 > Hour: 12:00 UTC (US AQI: 164.0)
The drill-down identifies the highest-average US AQI path from Bangladesh to Dhaka, then to 2026-11-13, 12-17, and hour 12:00 UTC. It shows where and when the forecast US AQI is highest within this small dataset. These values are model forecasts for a short period, not year-round measurements from ground-based air-quality sensors.


In [13]:
# Slice and Dice

def sql_in(values):
    """Make a SQL IN list from values obtained from Python."""
    return "(" + ", ".join(sql_text(value) for value in values) + ")"

# Helper to compare a SQL grouped result with the same pandas-filtered rows.
def check_slice_dice(sql_result, filtered_rows, group_cols, label):
    pandas_result = (
        filtered_rows.groupby(group_cols, as_index=False, dropna=False)
        .agg(
            AVG_PM2_5=("PM2_5", "mean"),
            HOURLY_ROWS=("PM2_5", "size")
        )
    )
    pandas_result["AVG_PM2_5"] = pandas_result["AVG_PM2_5"].round(2)

    sql_compare = sql_result.copy()
    pandas_compare = pandas_result.copy()

    # Use the same date text format on both sides for reliable matching.
    if "DATE_VALUE" in group_cols:
        sql_compare["DATE_VALUE"] = pd.to_datetime(
            sql_compare["DATE_VALUE"]
        ).dt.strftime("%Y-%m-%d")
        pandas_compare["DATE_VALUE"] = pd.to_datetime(
            pandas_compare["DATE_VALUE"]
        ).dt.strftime("%Y-%m-%d")

    merged = sql_compare.merge(
        pandas_compare,
        on=group_cols,
        how="outer",
        suffixes=("_SQL", "_PANDAS"),
        indicator=True
    )

    assert (merged["_merge"] == "both").all(), f"{label}: groups differ."
    assert (
        merged["HOURLY_ROWS_SQL"] == merged["HOURLY_ROWS_PANDAS"]
    ).all(), f"{label}: hourly row counts differ."
    assert (
        (merged["AVG_PM2_5_SQL"] - merged["AVG_PM2_5_PANDAS"]).abs() < 0.011
    ).all(), f"{label}: averages differ."

    print(f"{label}: SQL and pandas results match.")
    return pandas_result

# 1. SLICE on DATE_VALUE: fix the last date, keep all cities and all hour blocks.

last_date = date_text(air_df["DATE_VALUE"].max())

slice_date_df = run_sql(f"""
SELECT
    CITY,
    HOUR_BLOCK,
    ROUND(AVG(PM2_5), 2) AS AVG_PM2_5,
    COUNT(*) AS HOURLY_ROWS
FROM {DB}.MART.V_AIR_HOURLY
WHERE DATE_VALUE = TO_DATE({sql_text(last_date)})
GROUP BY CITY, HOUR_BLOCK
ORDER BY CITY, HOUR_BLOCK
""")

slice_date_rows = air_df[
    pd.to_datetime(air_df["DATE_VALUE"]).dt.strftime("%Y-%m-%d") == last_date
]
check_slice_dice(
    slice_date_df, slice_date_rows, ["CITY", "HOUR_BLOCK"], "Slice by date"
)
print("Slice date:", last_date)
display(slice_date_df)

# 2. SLICE on HOUR_BLOCK: calculate the block with the highest average PM2.5 in pandas, then fix that one block in SQL.

block_averages = air_df.groupby("HOUR_BLOCK")["PM2_5"].mean().sort_values(
    ascending=False
)
top_hour_block = block_averages.index[0]

slice_block_df = run_sql(f"""
SELECT
    CITY,
    DATE_VALUE,
    ROUND(AVG(PM2_5), 2) AS AVG_PM2_5,
    COUNT(*) AS HOURLY_ROWS
FROM {DB}.MART.V_AIR_HOURLY
WHERE HOUR_BLOCK = {sql_text(top_hour_block)}
GROUP BY CITY, DATE_VALUE
ORDER BY CITY, DATE_VALUE
""")

slice_block_rows = air_df[air_df["HOUR_BLOCK"] == top_hour_block]
check_slice_dice(
    slice_block_df, slice_block_rows, ["CITY", "DATE_VALUE"],
    "Slice by hour block"
)
print("Hour block selected:", top_hour_block)
display(slice_block_df)

# 3. DICE: choose two cities, two dates and two hour blocks. All selections come from values calculated in Python.

top_two_cities = (
    air_df.groupby("CITY")["PM2_5"].mean()
    .sort_values(ascending=False).head(2).index.tolist()
)

first_two_dates = sorted(
    pd.to_datetime(air_df["DATE_VALUE"]).dt.strftime("%Y-%m-%d").unique()
)[:2]

top_two_blocks = block_averages.head(2).index.tolist()

dice_df = run_sql(f"""
SELECT
    CITY,
    DATE_VALUE,
    HOUR_BLOCK,
    ROUND(AVG(PM2_5), 2) AS AVG_PM2_5,
    COUNT(*) AS HOURLY_ROWS
FROM {DB}.MART.V_AIR_HOURLY
WHERE CITY IN {sql_in(top_two_cities)}
  AND DATE_VALUE IN (
      TO_DATE({sql_text(first_two_dates[0])}),
      TO_DATE({sql_text(first_two_dates[1])})
  )
  AND HOUR_BLOCK IN {sql_in(top_two_blocks)}
GROUP BY CITY, DATE_VALUE, HOUR_BLOCK
ORDER BY CITY, DATE_VALUE, HOUR_BLOCK
""")

dice_rows = air_df[
    air_df["CITY"].isin(top_two_cities) &
    pd.to_datetime(air_df["DATE_VALUE"]).dt.strftime("%Y-%m-%d").isin(first_two_dates) &
    air_df["HOUR_BLOCK"].isin(top_two_blocks)
]
check_slice_dice(
    dice_df, dice_rows, ["CITY", "DATE_VALUE", "HOUR_BLOCK"], "Dice"
)

# 2 cities × 2 dates × 2 blocks = 8 groups.
# Each block contains six hourly observations: 8 × 6 = 48 rows.
assert len(dice_df) == 8, f"Expected 8 dice groups, got {len(dice_df)}."
assert int(dice_df["HOURLY_ROWS"].sum()) == 2 * 2 * 2 * 6

print("Cities selected:", top_two_cities)
print("Dates selected:", first_two_dates)
print("Hour blocks selected:", top_two_blocks)
display(dice_df)

# Print which dimensions are fixed and which contain selected values.
print(
    "Slice by date — fixed: DATE_VALUE =", last_date,
    "| chosen/remaining dimensions: all CITY values and all HOUR_BLOCK values."
)
print(
    "Slice by hour block — fixed: HOUR_BLOCK =", top_hour_block,
    "| chosen/remaining dimensions: all CITY values and all DATE_VALUE values."
)
print(
    "Dice — chosen values: 2 CITY values, 2 DATE_VALUE values and "
    "2 HOUR_BLOCK values (three dimensions restricted together)."
)

Slice by date: SQL and pandas results match.
Slice date: 2026-11-13


,CITY,HOUR_BLOCK,AVG_PM2_5,HOURLY_ROWS
0,Colombo,00-05,19.77,6
1,Colombo,06-11,18.73,6
2,Colombo,12-17,23.03,6
3,Colombo,18-23,19.73,6
4,Dhaka,00-05,72.40,6
5,Dhaka,06-11,34.80,6
6,Dhaka,12-17,81.72,6
7,Dhaka,18-23,70.37,6
8,Peshawar,00-05,79.27,6
9,Peshawar,06-11,49.87,6


Slice by hour block: SQL and pandas results match.
Hour block selected: 18-23


,CITY,DATE_VALUE,AVG_PM2_5,HOURLY_ROWS
0,Colombo,2026-11-11,6.20,6
1,Colombo,2026-11-12,23.27,6
2,Colombo,2026-11-13,19.73,6
3,Dhaka,2026-11-11,75.67,6
4,Dhaka,2026-11-12,96.00,6
5,Dhaka,2026-11-13,70.37,6
6,Peshawar,2026-11-11,57.02,6
7,Peshawar,2026-11-12,63.22,6
8,Peshawar,2026-11-13,89.68,6
9,Quetta,2026-11-11,33.07,6


Dice: SQL and pandas results match.
Cities selected: ['Peshawar', 'Dhaka']
Dates selected: ['2026-11-11', '2026-11-12']
Hour blocks selected: ['18-23', '12-17']


,CITY,DATE_VALUE,HOUR_BLOCK,AVG_PM2_5,HOURLY_ROWS
0,Dhaka,2026-11-11,12-17,58.25,6
1,Dhaka,2026-11-11,18-23,75.67,6
2,Dhaka,2026-11-12,12-17,75.85,6
3,Dhaka,2026-11-12,18-23,96.00,6
4,Peshawar,2026-11-11,12-17,76.50,6
5,Peshawar,2026-11-11,18-23,57.02,6
6,Peshawar,2026-11-12,12-17,79.07,6
7,Peshawar,2026-11-12,18-23,63.22,6


Slice by date — fixed: DATE_VALUE = 2026-11-13 | chosen/remaining dimensions: all CITY values and all HOUR_BLOCK values.
Slice by hour block — fixed: HOUR_BLOCK = 18-23 | chosen/remaining dimensions: all CITY values and all DATE_VALUE values.
Dice — chosen values: 2 CITY values, 2 DATE_VALUE values and 2 HOUR_BLOCK values (three dimensions restricted together).


In [14]:
# Pivot (Rotate) and GROUP BY CUBE

# Use text dates for clear, consistent pivot column labels.
pivot_source = air_df.copy()
pivot_source["DATE_VALUE"] = pd.to_datetime(
    pivot_source["DATE_VALUE"]
).dt.strftime("%Y-%m-%d")

# 1. Pivot: cities are rows, dates are columns; All gives totals.

pivot_city_date = pd.pivot_table(
    pivot_source,
    index="CITY",
    columns="DATE_VALUE",
    values="PM2_5",
    aggfunc="mean",
    margins=True,
    margins_name="All"
).round(2)

print("Pivot: cities as rows and dates as columns")
display(pivot_city_date)

# 2. CUBE creates all combinations: (CITY, DATE), city totals, date totals and grand total.

cube_df = run_sql(f"""
SELECT
    CITY,
    DATE_VALUE,
    ROUND(AVG(PM2_5), 2) AS AVG_PM2_5,
    COUNT(*) AS HOURLY_ROWS,
    GROUPING(CITY) AS G_CITY,
    GROUPING(DATE_VALUE) AS G_DATE
FROM {DB}.MART.V_AIR_HOURLY
GROUP BY CUBE (CITY, DATE_VALUE)
""")

number_of_cities = air_df["CITY"].nunique()
number_of_dates = air_df["DATE_VALUE"].nunique()
number_of_city_date_pairs = len(
    air_df[["CITY", "DATE_VALUE"]].drop_duplicates()
)
expected_cube_rows = (
    1 + number_of_cities + number_of_dates + number_of_city_date_pairs
)
assert len(cube_df) == expected_cube_rows

print("CUBE rows:", len(cube_df))
display(cube_df)

# 3. Every CUBE row must match a cell or total in the pivot. GROUPING flags tell which dimension was rolled up.

cube_df["DATE_LABEL"] = pd.to_datetime(
    cube_df["DATE_VALUE"]
).dt.strftime("%Y-%m-%d")

for _, row in cube_df.iterrows():
    if row["G_CITY"] == 0 and row["G_DATE"] == 0:
        # A city/date cell.
        pivot_value = pivot_city_date.loc[row["CITY"], row["DATE_LABEL"]]
    elif row["G_CITY"] == 0 and row["G_DATE"] == 1:
        # A city total across all dates: the All column.
        pivot_value = pivot_city_date.loc[row["CITY"], "All"]
    elif row["G_CITY"] == 1 and row["G_DATE"] == 0:
        # A date total across all cities: the All row.
        pivot_value = pivot_city_date.loc["All", row["DATE_LABEL"]]
    else:
        # Grand total: bottom-right All cell.
        pivot_value = pivot_city_date.loc["All", "All"]

    assert abs(float(row["AVG_PM2_5"]) - float(pivot_value)) < 0.011

print("Every GROUP BY CUBE average matches its pivot cell or total.")

# 4. Rotate the pivot: dates become rows and cities become columns.

pivot_date_city = pd.pivot_table(
    pivot_source,
    index="DATE_VALUE",
    columns="CITY",
    values="PM2_5",
    aggfunc="mean",
    margins=True,
    margins_name="All"
).round(2)

pd.testing.assert_frame_equal(pivot_date_city, pivot_city_date.T)
print("Rotated pivot (dates as rows and cities as columns):")
display(pivot_date_city)
print("Rotation check passed: pivot_date_city equals pivot_city_date.T.")

# 5. Pivot another measure: mean US_AQI by hour block/country. Find the highest cell and verify that exact cell in SQL.

pivot_block_country = pd.pivot_table(
    air_df,
    index="HOUR_BLOCK",
    columns="COUNTRY",
    values="US_AQI",
    aggfunc="mean"
).round(2)

print("Mean US AQI by hour block and country:")
display(pivot_block_country)

highest_cell = pivot_block_country.stack().idxmax()
highest_block = highest_cell[0]
highest_country = highest_cell[1]
highest_pivot_aqi = float(pivot_block_country.loc[highest_block, highest_country])

sql_cell_df = run_sql(f"""
SELECT ROUND(AVG(US_AQI), 2) AS AVG_US_AQI
FROM {DB}.MART.V_AIR_HOURLY
WHERE HOUR_BLOCK = {sql_text(highest_block)}
  AND COUNTRY = {sql_text(highest_country)}
""")

highest_sql_aqi = float(sql_cell_df.iloc[0]["AVG_US_AQI"])
assert abs(highest_pivot_aqi - highest_sql_aqi) < 0.011

print(
    f"Highest pivot cell: {highest_block}, {highest_country} "
    f"= {highest_pivot_aqi}; SQL value = {highest_sql_aqi}"
)

# 6. Identify the OLAP operation required by each question. Each operation is used exactly once.

operation_map = {
    "Q1": "roll-up",
    "Q2": "drill-down",
    "Q3": "slice",
    "Q4": "dice",
    "Q5": "pivot"
}

assert set(operation_map.values()) == {
    "roll-up", "drill-down", "slice", "dice", "pivot"
}
print("Operation map:")
for question, operation in operation_map.items():
    print(question, ":", operation)

Pivot: cities as rows and dates as columns


DATE_VALUE,2026-11-11,2026-11-12,2026-11-13,All
CITY,,,,
Colombo,11.29,15.16,20.32,15.59
Dhaka,46.51,63.28,64.82,58.20
Peshawar,51.84,54.72,75.97,60.84
Quetta,26.94,27.04,12.85,22.28
All,34.15,40.05,43.49,39.23


CUBE rows: 20


,CITY,DATE_VALUE,AVG_PM2_5,HOURLY_ROWS,G_CITY,G_DATE
0,Peshawar,2026-11-11,51.84,24,0,0
1,Peshawar,2026-11-12,54.72,24,0,0
2,Peshawar,2026-11-13,75.97,24,0,0
3,Quetta,2026-11-11,26.94,24,0,0
4,Quetta,2026-11-12,27.04,24,0,0
5,Quetta,2026-11-13,12.85,24,0,0
6,Dhaka,2026-11-12,63.28,24,0,0
7,Colombo,2026-11-11,11.29,24,0,0
8,Dhaka,2026-11-13,64.82,24,0,0
9,Colombo,2026-11-13,20.32,24,0,0


Every GROUP BY CUBE average matches its pivot cell or total.
Rotated pivot (dates as rows and cities as columns):


CITY,Colombo,Dhaka,Peshawar,Quetta,All
DATE_VALUE,,,,,
2026-11-11,11.29,46.51,51.84,26.94,34.15
2026-11-12,15.16,63.28,54.72,27.04,40.05
2026-11-13,20.32,64.82,75.97,12.85,43.49
All,15.59,58.20,60.84,22.28,39.23


Rotation check passed: pivot_date_city equals pivot_city_date.T.
Mean US AQI by hour block and country:


COUNTRY,Bangladesh,Pakistan,Sri Lanka
HOUR_BLOCK,,,
00-05,143.00,109.75,62.44
06-11,143.72,111.14,64.00
12-17,146.50,111.61,64.50
18-23,141.61,112.67,63.67


Highest pivot cell: 12-17, Bangladesh = 146.5; SQL value = 146.5
Operation map:
Q1 : roll-up
Q2 : drill-down
Q3 : slice
Q4 : dice
Q5 : pivot
